In [1]:
import pandas as pd 
import sqlite3


In [3]:
df= pd.read_csv('../data/retail_sales.csv')

conn = sqlite3.connect(':memory:')
df.to_sql('retail_sales', conn, index=False, if_exists='replace')

1000

In [5]:
#group by + having
q1= """
SELECT
    "Product Category",
    COUNT(*) AS total_orders,
    ROUND(SUM("Total Amount")) AS total_revenue,
    ROUND(AVG("Total Amount")) AS avg_order_value
FROM retail_sales
GROUP BY "Product Category"
HAVING AVG("Total Amount") > 100
ORDER BY total_revenue DESC
"""

In [ ]:
# subquery
q2= """
SELECT "Customer ID","Total Amount"
FROM retail_sales
WHERE "Total Amount" > (SELECT AVG("Total Amount") FROM retail_sales)
ORDER BY "Total Amount" DESC
LIMIT 10
"""

print("\n Top spenders (above average)")
print(pd.read_sql(q2, conn))


 Top spenders (above average)
  Customer ID  Total Amount
0     CUST015          2000
1     CUST065          2000
2     CUST072          2000
3     CUST074          2000
4     CUST089          2000
5     CUST093          2000
6     CUST109          2000
7     CUST118          2000
8     CUST124          2000
9     CUST139          2000


In [7]:
# CTE
q3="""
WITH category_stats AS (
SELECT
       "Product Category",
       AVG("Total Amount") AS avg_spend,
       COUNT(*) AS total_orders
FROM retail_sales
GROUP BY "Product Category"
)
SELECT *
FROM category_stats
WHERE avg_spend > 100
"""
print("\n CTE: category stats")
print(pd.read_sql(q3, conn))


 CTE: category stats
  Product Category   avg_spend  total_orders
0           Beauty  467.475570           307
1         Clothing  443.247863           351
2      Electronics  458.786550           342


In [8]:
# window func

q4="""
SELECT
    "Customer ID",
    "Product Category",
    "Total Amount",
    RANK() OVER (
        PARTITION BY "Product Category"
        ORDER BY "Total Amount" DESC
    ) AS rank_in_category,
    ROUND(AVG("Total Amount") OVER (
        PARTITION BY "Product Category"
    )) AS category_avg
FROM retail_sales
LIMIT 15
"""
print("\n window functions : rank and category avg")
print(pd.read_sql(q4, conn))


 window functions : rank and category avg
   Customer ID Product Category  Total Amount  rank_in_category  category_avg
0      CUST074           Beauty          2000                 1         467.0
1      CUST093           Beauty          2000                 1         467.0
2      CUST139           Beauty          2000                 1         467.0
3      CUST257           Beauty          2000                 1         467.0
4      CUST281           Beauty          2000                 1         467.0
5      CUST447           Beauty          2000                 1         467.0
6      CUST480           Beauty          2000                 1         467.0
7      CUST503           Beauty          2000                 1         467.0
8      CUST577           Beauty          2000                 1         467.0
9      CUST592           Beauty          2000                 1         467.0
10     CUST743           Beauty          2000                 1         467.0
11     CUST808       

In [10]:
#case when 
q5="""
SELECT
    "Customer ID",
    "Total Amount",
    CASE
        WHEN "Total Amount" >= 1000 THEN 'High Spender'
        WHEN "Total Amount" >= 500 THEN 'Medium Spender'
        ELSE 'Low Spender'
        END AS customer_segment
FROM retail_sales
ORDER BY "Total Amount" DESC
LIMIT 10
"""
print("\n case when : customer segments")
print(pd.read_sql(q5, conn))


 case when : customer segments
  Customer ID  Total Amount customer_segment
0     CUST015          2000     High Spender
1     CUST065          2000     High Spender
2     CUST072          2000     High Spender
3     CUST074          2000     High Spender
4     CUST089          2000     High Spender
5     CUST093          2000     High Spender
6     CUST109          2000     High Spender
7     CUST118          2000     High Spender
8     CUST124          2000     High Spender
9     CUST139          2000     High Spender


SQL JOINS
──────────────────────────────────────────
INNER JOIN → only matching rows in both tables
LEFT JOIN  → all left rows + matches from right
RIGHT JOIN → matches from left + all right rows
OUTER JOIN → everything from both tables

FILTERING
──────────────────────────────────────────
WHERE  → filter rows BEFORE grouping
HAVING → filter groups AFTER grouping

SUBQUERY vs CTE
──────────────────────────────────────────
Subquery → query inside a query (hard to read)
CTE      → named subquery using WITH (clean + reusable)

WINDOW FUNCTIONS
──────────────────────────────────────────
ROW_NUMBER() → unique rank, no ties
RANK()       → same rank for ties, skips numbers
DENSE_RANK() → same rank for ties, no skip
LAG()        → value from previous row
LEAD()       → value from next row
PARTITION BY → reset calculation per group

CASE WHEN
──────────────────────────────────────────
CASE
    WHEN condition1 THEN result1
    WHEN condition2 THEN result2
    ELSE default_result
END